In [27]:
import numpy as np
# Pothan Tang, 7/3/25
# Code modified to simulate the Rutherford scattering of dark matter particles without the presence of a trap potential
# Deleted functions trap_force, FRF, FDC, AllDCFs, DM_force, gradients, compute_alpha
# Assumed that the motion is classical, so no quantum effects like phonon excitations considered 

# Constants
elementary_charge = 1.602e-19  # Charge of an electron (C)
k_coulomb = 8.988e9  # Coulomb's constant (N·m²/C²)
trap_coefficient = 1e3  # Coefficient for quadratic trap potential
mass_dm = 1.0e-27  # Mass of dark matter particle (kg)
hbar=1.055e-34,# reduced Planck constant

# Time step and duration
step = 1e-5  # Small time step for numerical integration (modified to be much smaller)
t = 0
time_limit = 1  # Stop condition (modified to be much smaller)

#Itai's Constants (removed)

# Trap radius
r_0 = 0.5

# Initial dark matter conditions
a_old = np.array([0.0, 0.0, 0.0])
v_old = np.array([-1, 0.0, 0.0])
#x as R[cosphicostheta, cosphisintheta,sinphi]

## test values
phi = 0  # Example angle for phi (45 degrees)
theta = 0.1 # Example angle for theta (90 degrees)


x_old = r_0 * np.array([
        np.cos(phi) * np.cos(theta),
        np.cos(phi) * np.sin(theta),
        np.sin(phi)
    ])

# Ions setup (modified to include only 1 ion)
ions = [
    np.array([0.0, 0.0, 0.0]),
]
ion_velocities = [np.array([0.0, 0.0, 0.0]) for _ in ions]

# Data storage
position_of_particle = []
velocity_of_particle = []
acceleration_of_particle = []

def append_items(a, vnew, xnew):
    acceleration_of_particle.append(a)
    velocity_of_particle.append(vnew)
    position_of_particle.append(xnew)

def electric_field(epsilon, r_charge, ion):
    q = epsilon * elementary_charge
    r = ion - r_charge
    magnitude = np.linalg.norm(r)
    if magnitude == 0:
        return np.array([0.0, 0.0, 0.0])
    r_hat = r / magnitude
    return k_coulomb * (q / magnitude**2) * r_hat

def total_e(epsilon, r_charge, ions):
    total_field = np.zeros(3)
    for ion in ions:
        total_field += electric_field(epsilon, r_charge, ion)
    return total_field

def update_ions(x_dm):
    # trap force deleted
    for i in range(len(ions)):
        E_ion = total_e(1, ions[i], [x_dm])
        F_ion = elementary_charge * E_ion
        F_Total = F_ion
        a_ion = F_Total / mass_dm
        ion_velocities[i] += a_ion * step
        ions[i] += ion_velocities[i] * step

# Main simulation loop
terminate = False
while not terminate:
    # Compute forces and update motion
    E_rt = total_e(1, x_old, ions)
    Fdm_electric = elementary_charge * E_rt
    a = Fdm_electric / mass_dm
    vnew = v_old + a * step
    xnew = x_old + vnew * step
    append_items(a, vnew, xnew)

    # Update ions if inside trap radius
    if np.linalg.norm(x_old) <= r_0:
        update_ions(x_old)

    # Advance time and update state
    x_old, v_old, a_old = xnew, vnew, a
    t += step

    # Termination condition
    if t > time_limit or np.linalg.norm(xnew) > 10 * r_0:
        terminate = True
print(f"Initial position: {position_of_particle[1]}")
print(f"Final position: {position_of_particle[-1]}")
print(f"Initial velocity: {velocity_of_particle[1]}")
print(f"Final velocity: {velocity_of_particle[-1]}")
print(f"Initial acceleration: {acceleration_of_particle[1]}")
print(f"Final acceleration: {acceleration_of_particle[-1]}")
print(f"Final ion positions: {ions}")

## Calculate the scattering angle
# The scattering angle is the angle between the initial and final velocity vectors.
initial_velocity_direction = velocity_of_particle[0] / np.linalg.norm(velocity_of_particle[0])
final_velocity_direction = velocity_of_particle[-1] / np.linalg.norm(velocity_of_particle[-1])

# Use the dot product formula: a . b = |a||b|cos(theta)
dot_product = np.dot(initial_velocity_direction, final_velocity_direction)
scattering_angle_rad = np.arccos(np.clip(dot_product, -1.0, 1.0)) # Clip to avoid numerical errors
scattering_angle_deg = np.degrees(scattering_angle_rad)

print(f"\nSimulated scattering angle: {scattering_angle_deg:.4f} degrees")



Initial position: [0.49748208 0.04991671 0.        ]
Final position: [ 0.14958895 -0.00253478  0.        ]
Initial velocity: [-1.00001836e+00 -1.84232809e-06  0.00000000e+00]
Final velocity: [-0.2250949   0.01227706  0.        ]
Initial acceleration: [-0.9181004  -0.09211915  0.        ]
Final acceleration: [-0.35639216  0.02444461  0.        ]
Final ion positions: [array([-0.65209687,  0.05245149,  0.        ])]

Simulated scattering angle: 3.1220 degrees
